# EquityVista — Task 12: large multi-ticker US forecaster

**Framing:** a *cross-sectional ranker / daily scanner*, not a price oracle. One fixed model scores every ticker's next-day log return from scale-free features; it is judged per trading day (IC, AUC, decile spread), never on pooled rows.

## Data
- Kaggle US stock-market CSVs (CC0), 24.2M raw rows / 5,884 tickers → after ETL **11.06M rows / 2,447 tickers** (see `data/processed/etl_report.txt`).
- Dates 1962 → **2020-04-01** (the data stops there; it is stale for today's market).
- **Survivorship bias:** the universe is whatever was listed in the dump; delisted losers are under-represented, and the penny screen (median price < \$5 or volume < 100k) removes the riskiest names.

## Leakage controls
- **Global calendar split** (not per-ticker %): train ≤ 2015, val 2016-17, test 2018-2020, with 60-trading-day gaps so no window/label crosses a boundary.
- **20% of tickers held out** and only used in `test_unseen`.
- **Scale-free features** (returns, vol, ratios, RSI, same-date cross-sectional ranks): no price levels, no per-ticker scaler. CNN-LSTM standardisation uses one global TRAIN-only mean/std.
- `features_us.test_no_lookahead` truncates the data and checks features are unchanged.

## Honest result (preview)
No model beats the baselines **after costs**. LightGBM has a small but statistically positive IC (~0.012-0.013) and +2.8 pp direction accuracy over the majority baseline at 1 day (both seen and unseen tickers); its 5-day IC is positive on seen tickers only. The CNN-LSTM has no edge. Every decile spread is negative after a 20 bps round trip.

In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np, pandas as pd, polars as pl

# Default: load existing artifacts. Flip to True to rebuild (ETL ~minutes, CNN-LSTM training ~hours).
REBUILD_DATA = False      # src/build_dataset.py -> data/processed/us_market.parquet
REBUILD_FEATURES = False  # src/features_us.py   -> features.parquet
RETRAIN_LGBM = False      # src/models_us/lgbm_model.py
RETRAIN_CNN = False       # src/models_us/cnn_lstm_model.py (heavy; we load models/cnn_lstm_us.keras)
REEVALUATE = False        # src/evaluate_us.py -> eval_us_results.csv / eval_us_summary.txt

def run(*mod):
    subprocess.run([sys.executable, *mod], cwd=ROOT, check=True)

if REBUILD_DATA: run('src/build_dataset.py')
if REBUILD_FEATURES: run('src/features_us.py')
if RETRAIN_LGBM: run('-m', 'src.models_us.lgbm_model')
if RETRAIN_CNN: run('-m', 'src.models_us.cnn_lstm_model')
if REEVALUATE: run('-m', 'src.evaluate_us')
print((ROOT / 'data/processed/etl_report.txt').read_text()[:900])

## Features and split
`features.parquet` holds 13 per-row features + 12 same-date ranks used by LightGBM, plus the targets `y` (next-day log return) and `y5` (next 5 days).

In [ ]:
from src.split_us import FEATS, make_splits
from src.models_us.lgbm_model import FEATURES, predict_lgbm

feat = pl.read_parquet(FEATS)
print(feat.shape, len(FEATURES), 'LGBM features')
sp = make_splits(feat)
pd.DataFrame({k: {'rows': ids.height, 'tickers': ids['ticker'].n_unique(),
                  'from': ids['date'].min(), 'to': ids['date'].max()}
              for k, (ids, y) in sp.items()}).T

## Models
- **LightGBM (primary):** two heads (`y`, `y5`), 63 leaves, early-stopped on the later-in-time validation split.
- **CNN-LSTM (comparison arm):** Conv1D×2 → LSTM(64), 60-day windows of 11 features, trained on a 300-ticker subsample for 3 epochs (CPU only). Loaded, not retrained.

In [ ]:
import lightgbm as lgb
booster = lgb.Booster(model_file=str(ROOT / 'models/lgbm_us_y.txt'))
booster.best_iteration = booster.current_iteration()

# Quick sanity re-score on a 300k-row random sample of the seen-ticker test period (the full per-day eval is below)
ids, y = sp['test_seen']
idx = np.random.default_rng(0).choice(ids.height, 300_000, replace=False)
pred = predict_lgbm(booster, ids[idx], feat)
print('pooled corr(pred, y) on sample:', np.corrcoef(pred, y[idx, 0])[0, 1].round(4), '(pooled rows are NOT the headline metric)')

import keras
cnn = keras.saving.load_model(ROOT / 'models/cnn_lstm_us.keras', compile=False)
cnn.summary()

## Evaluation (per-day, with 95% CI)
Unit = one trading day. Direction accuracy is always paired with the majority baseline on the same days. Decile spread = long top-decile / short bottom-decile of predicted return, minus a 20 bps round trip every rebalance.

In [ ]:
res = pd.read_csv(ROOT / 'data/processed/eval_us_results.csv')
res['cell'] = res.apply(lambda r: f"{r['mean']:+.4f} [{r['lo']:+.4f},{r['hi']:+.4f}]", axis=1)
keep = ['ic', 'auc', 'diracc', 'diracc_vs_majority', 'gross_bps', 'net_bps']
for (h, s), g in res[res.metric.isin(keep)].groupby(['horizon', 'split']):
    print(f'--- horizon {h}d | {s} | {int(g.n_days.max())} days')
    display(g.pivot(index='model', columns='metric', values='cell')[[m for m in keep if m in g.metric.values]])

In [ ]:
# Verdict lines produced by src/evaluate_us.py
txt = (ROOT / 'data/processed/eval_us_summary.txt').read_text()
print('\n'.join(l for l in txt.splitlines() if l.startswith('[') or l.startswith('   ')))

## Verdict
- **LightGBM:** 1-day mean daily IC ≈ +0.013 (seen) / +0.012 (unseen), CI excludes 0; direction accuracy +2.8 pp over majority. Real but tiny, and the net decile spread is **−14.8 / −12.7 bps** after costs → **not tradable**. 5-day IC is positive on seen tickers only.
- **CNN-LSTM:** IC and direction accuracy indistinguishable from zero edge. It does not beat the LightGBM baseline.
- **Baselines:** the 1-day reversal rule and zero/majority predictors also show no edge. The majority baseline is *not* 50%, because the up-fraction was ≈51% in the test period, which is why accuracy is always paired.
- Caveats: survivorship bias, data ends 2020-04, CIs are slightly optimistic (daily IC is mildly autocorrelated), costs are a flat 20 bps round trip.

## Using it on a live ticker
`src/predict_us.py` downloads ~2 years of data with `yfinance`, builds the same scale-free features on the ticker plus a reference basket of ~40 large caps (needed for `mkt_ret` and the `r_*` cross-sectional ranks, since a single ticker has no cross-section), then scores the last row with both fixed models. No per-ticker training.

```bash
python -m src.predict_us AAPL MSFT
```
Output is a next-day log-return view from each model. Treat it as a scanner signal, **educational, not financial advice**. Ranks are relative to the basket, not the 2,447-name training universe, so live scores are an approximation.

In [ ]:
LIVE = False  # needs internet + yfinance
if LIVE:
    from src.predict_us import predict_ticker, show
    for t in ['AAPL', 'MSFT']:
        try:
            show(predict_ticker(t))
        except Exception as e:
            print(t, 'failed:', e)